# LayerSkip：真正的逐层模型、自投机接受与回滚

**面试问题：LayerSkip 为什么需要中间层训练，服务时怎样让 draft 有用又不越过完整模型？**

## 回答主线

LayerSkip 不是给几组静态 logits 起名字，而是让同一个模型的中间层通过可训练出口产生可度量的候选。真实 LLM 可以让各层复用同一个 LM head，以保持词表空间和参数语义一致；本教学小网络为了分别观察三层出口的学习能力，明确使用三套独立辅助分类头，不虚称权重共享。早层计算少，往往先捕获主题等粗粒度信号；深层继续整合金额、风险、证据等细节，因此 target 质量通常更高。训练时可以给多个出口施加辅助交叉熵，但完整出口仍承担主要损失，避免只优化早退准确率。纯置信度早退会把“自信但错误”的 draft 直接提交，所以自投机方案必须由 target 验证：一致则接受，不一致就回滚到 target。评估至少同时报告各层准确率、draft 接受率、回滚样本和最终一致率，不能只打印一个布尔断言。下面手写 PyTorch 参数、逐层 forward、真实 backward 与参数更新，并在同一批业务请求上比较关键词基线、早层 draft 和完整模型。

## 真实案例：客服动作路由

案例包含 12 条脱敏请求，覆盖退款、物流和账号三类主题。每条记录有请求编号、主题、金额强度、风险分、证据是否充分、VIP 标记和延迟强度，目标动作是自动答复、查询物流、身份核验或人工审批。每个主题有三条常规请求和一条高风险请求，因此只识别主题的浅层模型会稳定犯错，而整合细节的深层模型可以修正。数据是结构与真实路由表一致的离线教学样本，不含真实用户信息；模型在这 12 条训练样本上的结果只解释 LayerSkip 机制，不能外推为线上泛化或吞吐收益。

### 输入预览：12 条有业务语义的请求

In [1]:
import warnings  # 导入告警控制工具以屏蔽当前教学环境的第三方依赖提示。
warnings.filterwarnings("ignore", message="The pynvml package is deprecated.*", category=FutureWarning)  # 只过滤 PyTorch 导入时由环境包触发的已知提示。
import torch  # 导入 PyTorch 基础张量与自动微分能力。
torch.set_num_threads(1)  # 固定单线程执行以减少小实验的调度噪声。
torch.manual_seed(2603)  # 固定参数初始化以保证训练曲线和输出可复现。
feature_names = ["退款主题", "物流主题", "账号主题", "金额强度", "风险分", "证据充分", "VIP", "延迟强度"]  # 定义八个可读输入字段。
action_names = ["自动答复", "查询物流", "身份核验", "人工审批"]  # 定义四个客服动作类别。
records = [  # 构造十二条脱敏客服路由记录。
    {"id": "RF-101", "summary": "小额退款且凭证齐全", "features": [1, 0, 0, 0.10, 0.10, 1, 0, 0.00], "label": 0},  # 常规退款应自动答复时效。
    {"id": "RF-102", "summary": "VIP 小额退款咨询", "features": [1, 0, 0, 0.30, 0.20, 1, 1, 0.00], "label": 0},  # VIP 不改变低风险退款路线。
    {"id": "RF-103", "summary": "中额退款且凭证齐全", "features": [1, 0, 0, 0.50, 0.30, 1, 0, 0.00], "label": 0},  # 中等金额仍在自动答复边界内。
    {"id": "RF-104", "summary": "大额高风险且无凭证", "features": [1, 0, 0, 0.95, 0.90, 0, 0, 0.00], "label": 3},  # 同为退款主题但必须人工审批。
    {"id": "LG-201", "summary": "普通包裹轻微延迟", "features": [0, 1, 0, 0.00, 0.10, 1, 0, 0.20], "label": 1},  # 低风险物流请求直接查单。
    {"id": "LG-202", "summary": "VIP 包裹轻微延迟", "features": [0, 1, 0, 0.00, 0.20, 1, 1, 0.30], "label": 1},  # VIP 常规延迟仍先查询物流。
    {"id": "LG-203", "summary": "包裹中度延迟有轨迹", "features": [0, 1, 0, 0.00, 0.30, 1, 0, 0.50], "label": 1},  # 有轨迹的中度延迟可自动查单。
    {"id": "LG-204", "summary": "长期停滞且轨迹缺失", "features": [0, 1, 0, 0.00, 0.85, 0, 1, 0.95], "label": 3},  # 高风险长期停滞需要人工介入。
    {"id": "AC-301", "summary": "低风险登录失败", "features": [0, 0, 1, 0.00, 0.10, 1, 0, 0.00], "label": 2},  # 常规账号问题进入身份核验。
    {"id": "AC-302", "summary": "VIP 密码重置", "features": [0, 0, 1, 0.00, 0.20, 1, 1, 0.00], "label": 2},  # VIP 也不能跳过身份核验。
    {"id": "AC-303", "summary": "异地登录但证据充分", "features": [0, 0, 1, 0.00, 0.30, 1, 0, 0.00], "label": 2},  # 轻度风险仍可走标准核验。
    {"id": "AC-304", "summary": "高风险接管且无证据", "features": [0, 0, 1, 0.00, 0.95, 0, 0, 0.00], "label": 3},  # 同为账号主题但必须人工审批。
]  # 完成十二条结构化业务样本。
features = torch.tensor([record["features"] for record in records], dtype=torch.float32)  # 把可读字段转换成模型输入矩阵。
labels = torch.tensor([record["label"] for record in records], dtype=torch.long)  # 把目标动作转换成类别索引。
print("输入形状：", tuple(features.shape), "字段：", feature_names)  # 展示真实输入合同和张量形状。
print("请求 | 场景摘要 | 目标动作 | 风险 | 证据")  # 输出逐请求预览表头。
for record in records:  # 逐条展示学习者可读的业务样本。
    print(f"{record['id']} | {record['summary']} | {action_names[record['label']]} | {record['features'][4]:.2f} | {record['features'][5]}")  # 输出请求语义与关键门禁字段。

输入形状： (12, 8) 字段： ['退款主题', '物流主题', '账号主题', '金额强度', '风险分', '证据充分', 'VIP', '延迟强度']
请求 | 场景摘要 | 目标动作 | 风险 | 证据
RF-101 | 小额退款且凭证齐全 | 自动答复 | 0.10 | 1
RF-102 | VIP 小额退款咨询 | 自动答复 | 0.20 | 1
RF-103 | 中额退款且凭证齐全 | 自动答复 | 0.30 | 1
RF-104 | 大额高风险且无凭证 | 人工审批 | 0.90 | 0
LG-201 | 普通包裹轻微延迟 | 查询物流 | 0.10 | 1
LG-202 | VIP 包裹轻微延迟 | 查询物流 | 0.20 | 1
LG-203 | 包裹中度延迟有轨迹 | 查询物流 | 0.30 | 1
LG-204 | 长期停滞且轨迹缺失 | 人工审批 | 0.85 | 0
AC-301 | 低风险登录失败 | 身份核验 | 0.10 | 1
AC-302 | VIP 密码重置 | 身份核验 | 0.20 | 1
AC-303 | 异地登录但证据充分 | 身份核验 | 0.30 | 1
AC-304 | 高风险接管且无证据 | 人工审批 | 0.95 | 0


## Baseline 基线：只看主题关键词

最便宜的规则把退款、物流、账号分别固定映射到自动答复、查询物流、身份核验。它能处理每类常规请求，却看不到金额、风险、证据和延迟，因此会误放三条高风险请求。这个基线与后续模型使用完全相同的 12 条样本和准确率口径。

In [2]:
topic_to_action = torch.tensor([0, 1, 2], dtype=torch.long)  # 定义退款、物流、账号的固定主题动作。
topic_indices = torch.argmax(features[:, :3], dim=1)  # 从前三个 one-hot 字段识别请求主题。
baseline_predictions = topic_to_action[topic_indices]  # 根据主题规则生成十二条基线预测。
baseline_accuracy = float((baseline_predictions == labels).float().mean())  # 计算同一数据上的基线准确率。
baseline_wrong_ids = [records[index]["id"] for index in range(len(records)) if baseline_predictions[index] != labels[index]]  # 收集被主题规则误放的请求。
print(f"主题规则准确率：{baseline_accuracy:.1%}")  # 输出可与逐层模型直接比较的指标。
print("主题规则错误请求：", baseline_wrong_ids)  # 展示错误集中在三个高风险反例。
print("原因：主题相同不代表风险动作相同，规则没有读取金额、证据和延迟字段。")  # 解释基线的结构性缺陷。

主题规则准确率：75.0%
主题规则错误请求： ['RF-104', 'LG-204', 'AC-304']
原因：主题相同不代表风险动作相同，规则没有读取金额、证据和延迟字段。


## 核心实现：手写三层共享模型与三个出口

第一层只形成主题表示，模拟早层先捕获粗粒度语义；第二层开始读取五个业务细节，第三层通过残差非线性继续整合。三个出口是三套独立、实际参与训练的辅助分类头，不是预先填好的 logits，也不宣称共享权重；这种设计便于逐层观察，但真实 LLM 也可以复用同一个 LM head。为了让计算过程透明，下面不用现成 Transformer、Trainer 或 EarlyExit 包，只使用 Parameter、矩阵乘法、tanh、ReLU 和自动微分。

In [3]:
class LayerSkipRouter(torch.nn.Module):  # 定义带三个真实出口的逐层路由网络。
    def __init__(self, hidden_dim=12, class_count=4):  # 初始化隐藏维度和动作类别数。
        super().__init__()  # 注册 PyTorch 模块的参数管理能力。
        self.topic_weight = torch.nn.Parameter(torch.randn(3, hidden_dim) * 0.15)  # 初始化主题字段到第一层表示的权重。
        self.topic_bias = torch.nn.Parameter(torch.zeros(hidden_dim))  # 初始化第一层偏置。
        self.detail_weight = torch.nn.Parameter(torch.randn(5, hidden_dim) * 0.15)  # 初始化五个细节字段的投影权重。
        self.block1_weight = torch.nn.Parameter(torch.randn(hidden_dim, hidden_dim) * 0.15)  # 初始化第二层主题变换权重。
        self.block1_bias = torch.nn.Parameter(torch.zeros(hidden_dim))  # 初始化第二层偏置。
        self.block2_weight = torch.nn.Parameter(torch.randn(hidden_dim, hidden_dim) * 0.15)  # 初始化第三层残差变换权重。
        self.block2_bias = torch.nn.Parameter(torch.zeros(hidden_dim))  # 初始化第三层偏置。
        self.exit1_weight = torch.nn.Parameter(torch.randn(hidden_dim, class_count) * 0.15)  # 初始化第一层 draft 输出头。
        self.exit1_bias = torch.nn.Parameter(torch.zeros(class_count))  # 初始化第一层输出偏置。
        self.exit2_weight = torch.nn.Parameter(torch.randn(hidden_dim, class_count) * 0.15)  # 初始化第二层辅助输出头。
        self.exit2_bias = torch.nn.Parameter(torch.zeros(class_count))  # 初始化第二层输出偏置。
        self.exit3_weight = torch.nn.Parameter(torch.randn(hidden_dim, class_count) * 0.15)  # 初始化完整 target 输出头。
        self.exit3_bias = torch.nn.Parameter(torch.zeros(class_count))  # 初始化完整输出偏置。
    def forward(self, batch_features):  # 实现从业务字段到三个逐层 logits 的前向传播。
        topic_features = batch_features[:, :3]  # 取出三个主题 one-hot 字段供早层使用。
        detail_features = batch_features[:, 3:]  # 取出金额、风险、证据、VIP 和延迟细节。
        hidden1 = torch.tanh(topic_features @ self.topic_weight + self.topic_bias)  # 计算只理解主题的第一层表示。
        hidden2 = torch.tanh(hidden1 @ self.block1_weight + detail_features @ self.detail_weight + self.block1_bias)  # 在第二层融合主题与风险细节。
        residual_update = torch.relu(hidden2 @ self.block2_weight + self.block2_bias)  # 计算第三层非线性残差更新。
        hidden3 = torch.tanh(hidden2 + residual_update)  # 得到完整模型用于最终判断的深层表示。
        logits1 = hidden1 @ self.exit1_weight + self.exit1_bias  # 从第一层产生可训练 draft logits。
        logits2 = hidden2 @ self.exit2_weight + self.exit2_bias  # 从第二层产生辅助 logits。
        logits3 = hidden3 @ self.exit3_weight + self.exit3_bias  # 从第三层产生权威 target logits。
        return [logits1, logits2, logits3], [hidden1, hidden2, hidden3]  # 返回所有出口和中间状态供训练与解释。
model = LayerSkipRouter()  # 实例化真正执行 forward 的逐层模型。
initial_logits, initial_hidden = model(features)  # 在训练前运行一次完整前向以验证计算图。
parameter_count = sum(parameter.numel() for parameter in model.parameters())  # 统计手写模型的全部可训练参数。
print("可训练参数量：", parameter_count)  # 展示本例确实包含模型参数而非静态分数表。
print("逐层 hidden 形状：", [tuple(hidden.shape) for hidden in initial_hidden])  # 展示三层中间表示的真实张量形状。
print("逐层 logits 形状：", [tuple(logits.shape) for logits in initial_logits])  # 展示每个出口都对十二条请求产生四类分数。

可训练参数量： 576
逐层 hidden 形状： [(12, 12), (12, 12), (12, 12)]
逐层 logits 形状： [(12, 4), (12, 4), (12, 4)]


### 真实训练：辅助出口损失、完整出口主损失与 backward

第一层损失权重较小，因为它只能看到主题；第二层和第三层逐步承担更多监督。训练循环显式执行 forward、手写稳定交叉熵、backward、参数更新和梯度清零。打印的损失与三层准确率来自真实计算。

In [4]:
def manual_cross_entropy(logits, target_labels):  # 手写稳定的多分类交叉熵。
    log_probabilities = logits - torch.logsumexp(logits, dim=1, keepdim=True)  # 用 log-sum-exp 得到稳定对数概率。
    row_indices = torch.arange(target_labels.shape[0])  # 构造每条请求的行索引。
    return -log_probabilities[row_indices, target_labels].mean()  # 取目标类别负对数概率并求平均。
learning_rate = 0.08  # 设置适合全批量小数据的学习率。
loss_weights = [0.15, 0.35, 1.00]  # 让完整出口承担主要监督同时训练两个早层出口。
training_trace = []  # 保存关键训练步的损失和逐层准确率。
for step in range(601):  # 执行六百次真实前向与反向更新。
    exit_logits, _ = model(features)  # 运行当前参数下的三层前向传播。
    exit_losses = [manual_cross_entropy(logits, labels) for logits in exit_logits]  # 分别计算三个出口的真实交叉熵。
    total_loss = sum(weight * loss for weight, loss in zip(loss_weights, exit_losses))  # 按出口重要性合并训练目标。
    total_loss.backward()  # 通过三个出口把梯度传播到共享层和输出头。
    if step in {0, 100, 200, 400, 600}:  # 在固定步数保存可读训练快照。
        exit_accuracies = [float((logits.argmax(dim=1) == labels).float().mean()) for logits in exit_logits]  # 计算三个出口的训练集准确率。
        training_trace.append((step, float(total_loss.detach()), exit_accuracies))  # 保存损失与逐层指标。
    with torch.no_grad():  # 关闭参数更新阶段的梯度记录。
        for parameter in model.parameters():  # 遍历手写模型的全部可训练参数。
            parameter -= learning_rate * parameter.grad  # 使用最基础的梯度下降更新当前参数。
            parameter.grad.zero_()  # 清空梯度避免下一步错误累加。
print("step | total_loss | layer1 | layer2 | target")  # 输出真实训练过程表头。
for step, loss_value, accuracies in training_trace:  # 逐个展示保存的训练快照。
    print(f"{step:4d} | {loss_value:10.4f} | {accuracies[0]:6.1%} | {accuracies[1]:6.1%} | {accuracies[2]:6.1%}")  # 展示损失下降和深层能力提升。

step | total_loss | layer1 | layer2 | target
   0 |     2.0364 |   8.3% |  25.0% |  33.3%
 100 |     0.7081 |  75.0% |  91.7% | 100.0%
 200 |     0.3133 |  75.0% | 100.0% | 100.0%
 400 |     0.1688 |  75.0% | 100.0% | 100.0%
 600 |     0.1336 |  75.0% | 100.0% | 100.0%


## 结果解读：draft 接受、target 验证与回滚账本

训练后的第一层仍只能按主题做多数决策，所以它会把三条高风险请求当作常规请求；完整出口已经读到风险细节。自投机协议先产生 layer-1 draft，再由完整 target 验证：置信度达到阈值且类别一致才接受，否则回滚到 target。这里为了建立正确性 oracle 会计算全部出口；真实吞吐还需多 token draft、批量 target 验证和 kernel 级计时。

In [5]:
with torch.no_grad():  # 在评估阶段关闭梯度以得到稳定推理结果。
    evaluated_logits, evaluated_hidden = model(features)  # 对十二条请求运行三个真实模型出口。
    draft_probabilities = torch.softmax(evaluated_logits[0], dim=1)  # 把第一层 logits 转换成 draft 概率。
    draft_confidences, draft_predictions = draft_probabilities.max(dim=1)  # 提取 draft 类别和置信度。
    second_predictions = evaluated_logits[1].argmax(dim=1)  # 提取第二层预测用于逐层比较。
    target_predictions = evaluated_logits[2].argmax(dim=1)  # 提取完整模型的权威预测。
confidence_threshold = 0.60  # 设置允许候选进入接受判断的教学阈值。
accepted_mask = (draft_confidences >= confidence_threshold) & (draft_predictions == target_predictions)  # 只有高置信且 target 同意才接受 draft。
served_predictions = torch.where(accepted_mask, draft_predictions, target_predictions)  # 对不一致候选回滚到完整模型结果。
layer_accuracies = [float((logits.argmax(dim=1) == labels).float().mean()) for logits in evaluated_logits]  # 计算三个真实出口的准确率。
acceptance_rate = float(accepted_mask.float().mean())  # 计算 draft 被 target 接受的样本比例。
final_match_rate = float((served_predictions == target_predictions).float().mean())  # 验证最终提交结果与 target 一致。
print("请求 | 目标 | layer1/conf | layer2 | target | 自投机动作")  # 输出逐请求接受与回滚账本表头。
for index, record in enumerate(records):  # 遍历十二条业务请求展示模型过程。
    decision = "接受draft" if accepted_mask[index] else "回滚target"  # 把布尔门禁转换成可读动作。
    print(f"{record['id']} | {action_names[labels[index]]} | {action_names[draft_predictions[index]]}/{draft_confidences[index]:.3f} | {action_names[second_predictions[index]]} | {action_names[target_predictions[index]]} | {decision}")  # 展示真实逐层预测和提交决定。
print("逐层准确率：", [f"{accuracy:.1%}" for accuracy in layer_accuracies])  # 汇总早层到完整出口的质量变化。
print(f"draft 接受率={acceptance_rate:.1%}，最终 target 一致率={final_match_rate:.1%}")  # 同时报告效率代理和正确性指标。

请求 | 目标 | layer1/conf | layer2 | target | 自投机动作
RF-101 | 自动答复 | 自动答复/0.686 | 自动答复 | 自动答复 | 接受draft
RF-102 | 自动答复 | 自动答复/0.686 | 自动答复 | 自动答复 | 接受draft
RF-103 | 自动答复 | 自动答复/0.686 | 自动答复 | 自动答复 | 接受draft
RF-104 | 人工审批 | 自动答复/0.686 | 人工审批 | 人工审批 | 回滚target
LG-201 | 查询物流 | 查询物流/0.699 | 查询物流 | 查询物流 | 接受draft
LG-202 | 查询物流 | 查询物流/0.699 | 查询物流 | 查询物流 | 接受draft
LG-203 | 查询物流 | 查询物流/0.699 | 查询物流 | 查询物流 | 接受draft
LG-204 | 人工审批 | 查询物流/0.699 | 人工审批 | 人工审批 | 回滚target
AC-301 | 身份核验 | 身份核验/0.676 | 身份核验 | 身份核验 | 接受draft
AC-302 | 身份核验 | 身份核验/0.676 | 身份核验 | 身份核验 | 接受draft
AC-303 | 身份核验 | 身份核验/0.676 | 身份核验 | 身份核验 | 接受draft
AC-304 | 人工审批 | 身份核验/0.676 | 人工审批 | 人工审批 | 回滚target
逐层准确率： ['75.0%', '100.0%', '100.0%']
draft 接受率=75.0%，最终 target 一致率=100.0%


## 失败案例与修正：高置信不等于可直接提交

错误实现只检查 layer-1 置信度，超过 0.60 就直接提交。由于早层按主题学到了约 3:1 的多数规律，它会很自信地误放 RF-104、LG-204 和 AC-304。修正方案不是调一个更漂亮的阈值，而是保留 target 的提交权；target 不同意时必须回滚。

In [6]:
naive_predictions = torch.where(draft_confidences >= confidence_threshold, draft_predictions, target_predictions)  # 模拟只看置信度就提交的危险实现。
naive_wrong_indices = [index for index in range(len(records)) if naive_predictions[index] != labels[index]]  # 找出天真早退造成的错误请求。
safe_wrong_indices = [index for index in range(len(records)) if served_predictions[index] != labels[index]]  # 检查 target 验证后的最终错误请求。
print("错误实现误放数量：", len(naive_wrong_indices))  # 输出高置信直接提交的真实错误数。
for index in naive_wrong_indices:  # 逐条展示不能被总准确率掩盖的高风险错误。
    print(f"失败请求={records[index]['id']}，draft={action_names[draft_predictions[index]]}，置信度={draft_confidences[index]:.3f}，target={action_names[target_predictions[index]]}")  # 展示错误 draft 如何被完整模型修正。
print("修正后错误数量：", len(safe_wrong_indices))  # 展示 verifier 回滚后的最终结果。
print("修正规则：draft 只能提议，完整 target 才拥有提交权。")  # 明确可带到面试中的安全不变量。

错误实现误放数量： 3
失败请求=RF-104，draft=自动答复，置信度=0.686，target=人工审批
失败请求=LG-204，draft=查询物流，置信度=0.699，target=人工审批
失败请求=AC-304，draft=身份核验，置信度=0.676，target=人工审批
修正后错误数量： 0
修正规则：draft 只能提议，完整 target 才拥有提交权。


### 生产差距

本例是全批量训练的四分类小网络，第一层只读主题是为了让逐层能力差异可解释；真实 LLM 每层都处理完整 token 表示。生产实现还要在预训练阶段加入 layer dropout 与共享 LM head 辅助损失，校准不同长度和领域的置信度，并实现多 token draft、并行 target verification、KV/hidden state 复用和真实 GPU 吞吐评测。接受率不能脱离任务质量、回滚成本和 batch 形态单独宣传。

In [7]:
layerskip_contract = {  # 构造训练与服务共同版本化的 LayerSkip 合同。
    "draft_exit": 1,  # 指定第一层出口负责产生候选。
    "target_exit": 3,  # 指定第三层完整出口拥有最终判断权。
    "confidence_threshold": confidence_threshold,  # 保存经过校准的候选置信度阈值。
    "commit_authority": "target_verifier",  # 明确禁止 draft 绕过完整验证直接提交。
    "metrics": ["exit_accuracy", "acceptance_rate", "rollback_count", "target_match"],  # 定义发布前必须共同观察的指标。
}  # 完成可审计的逐层服务合同。
rollback_ids = [records[index]["id"] for index in range(len(records)) if not accepted_mask[index]]  # 收集真实发生回滚的请求编号。
print("LayerSkip 发布合同：", layerskip_contract)  # 展示模型制品与服务策略必须一起版本化。
print("本批回滚请求：", rollback_ids)  # 展示回滚不是抽象枚举而是可定位事件。
print("教学实验边界：这些数字只来自 12 条训练样本，不代表线上准确率或 GPU 加速比。")  # 防止把受控实验冒充生产收益。

LayerSkip 发布合同： {'draft_exit': 1, 'target_exit': 3, 'confidence_threshold': 0.6, 'commit_authority': 'target_verifier', 'metrics': ['exit_accuracy', 'acceptance_rate', 'rollback_count', 'target_match']}
本批回滚请求： ['RF-104', 'LG-204', 'AC-304']
教学实验边界：这些数字只来自 12 条训练样本，不代表线上准确率或 GPU 加速比。


## 最小回归测试

断言只保护模型确实训练、深层优于浅层、危险反例存在、verifier 能回滚以及最终提交权不被破坏；学习证据是前面的输入、训练曲线和逐请求账本。

In [8]:
assert parameter_count > 0  # 保证案例包含真正可训练的 PyTorch 参数。
assert layer_accuracies[0] == baseline_accuracy  # 保证第一层真实复现只看主题的能力上限。
assert layer_accuracies[2] > layer_accuracies[0]  # 保证完整出口通过细节计算修正浅层错误。
assert len(naive_wrong_indices) >= 1  # 保证高置信直接提交的失败案例真实发生。
assert len(safe_wrong_indices) == 0  # 保证 target verifier 已修正本批全部错误 draft。
assert torch.equal(served_predictions, target_predictions)  # 保证任何最终提交都与完整模型一致。
print("回归测试通过：真实模型、逐层训练、错误 draft、回滚和 target 提交权均成立。")  # 汇总少量关键不变量。

回归测试通过：真实模型、逐层训练、错误 draft、回滚和 target 提交权均成立。
